In [1]:
# 1. Clean up the broken packages
!pip uninstall -y transformers peft accelerate

# 2. Freshly install the upgraded libraries
!pip install -q -U torch transformers peft accelerate

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.0 MB/s eta 0:00:00


In [3]:
!pip install -q torch transformers peft
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

print("Loading base model...")
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float16, device_map="cpu")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("Applying your custom adapters...")
model = PeftModel.from_pretrained(base_model, "/content/adapters")

print("Merging into a single brain...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("/content/merged_model")
tokenizer.save_pretrained("/content/merged_model")

print("✅ Merge complete! Model is ready for compression.")

Loading base model...


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Applying your custom adapters...


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

Merging into a single brain...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merge complete! Model is ready for compression.


In [4]:
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && make -j
!pip install -r llama.cpp/requirements.txt

Cloning into 'llama.cpp'...
remote: Enumerating objects: 82345, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 82345 (delta 25), reused 6 (delta 6), pack-reused 82283 (from 2)
Receiving objects: 100% (82345/82345), 312.36 MiB | 25.70 MiB/s, done.
Resolving deltas: 100% (59219/59219), done.
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment


In [5]:
# 1. Install the missing tokenizer libraries
!pip install -q tokenizers sentencepiece

# 2. Download the missing tokenizer.model directly into your merged folder
!wget https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer.model -O /content/merged_model/tokenizer.model

--2026-03-07 12:04:54--  https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer.model
Resolving huggingface.co (huggingface.co)... 18.238.109.52, 18.238.109.102, 18.238.109.121, ...
Connecting to huggingface.co (huggingface.co)|18.238.109.52|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/658fb85235c41262d661dc48/91bf184ab12793d0754344f9095332759432e666320cc6c07f637af50e36db6f?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27tokenizer.model%3B+filename%3D%22tokenizer.model%22%3B&Expires=1772888694&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzcyODg4Njk0fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU4ZmI4NTIzNWM0MTI2MmQ2NjFkYzQ4LzkxYmYxODRhYjEyNzkzZDA3NTQzNDRmOTA5NTMzMjc1OTQzMmU2NjYzMjBjYzZjMDdmNjM3YWY1MGUzNmRiNmZcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=Zp-OnHMTKm4OnB90S3lOT8GsMHTbvLtZnQapJEQGwU

In [6]:
!python llama.cpp/convert_hf_to_gguf.py /content/merged_model --outfile /content/model-f16.gguf --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.we

In [7]:
# Cell 4 — The Shrink Ray (Quantize to 4-bit using CMake build)

# 1. Build llama.cpp using the modern CMake method
!cd /content/llama.cpp && cmake -B build && cmake --build build --config Release -j 4

# 2. Compress the FP16 model into a 4-bit GGUF model
!/content/llama.cpp/build/bin/llama-quantize /content/model-f16.gguf /content/model-q4_k_m.gguf q4_k_m

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [9]:
!/content/llama.cpp/build/bin/llama-quantize /content/model-f16.gguf /content/model-q8_0.gguf q8_0

main: build = 8233 (c5a778891)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/model-f16.gguf' to '/content/model-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 31 key-value pairs and 201 tensors from /content/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - 

In [13]:
!pip install -q -U bitsandbytes accelerate

In [14]:
!pip uninstall -y torchvision torchaudio

Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu


In [10]:
!pip install "triton==3.2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.2/253.2 MB 5.6 MB/s eta 0:00:00


In [1]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("1. Quantizing and saving Hugging Face INT8...")
bnb_config_8 = BitsAndBytesConfig(load_in_8bit=True)
model_8bit = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_config_8,
    device_map="auto"
)
model_8bit.save_pretrained("/content/quantized/model-int8")

# Delete the 8-bit model from RAM so Colab doesn't crash
del model_8bit
torch.cuda.empty_cache()
gc.collect()

print("2. Quantizing and saving Hugging Face INT4...")
bnb_config_4 = BitsAndBytesConfig(load_in_4bit=True)
model_4bit = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_config_4,
    device_map="auto"
)
model_4bit.save_pretrained("/content/quantized/model-int4")

del model_4bit
torch.cuda.empty_cache()
gc.collect()

print("✅ Success! You now have Hugging Face INT8 and INT4 folders.")

1. Quantizing and saving Hugging Face INT8...
2. Quantizing and saving Hugging Face INT4...
✅ Success! You now have Hugging Face INT8 and INT4 folders.


In [2]:
# Zip the 8-bit model folder
!zip -r /content/model-int8.zip /content/quantized/model-int8

# Zip the 4-bit model folder
!zip -r /content/model-int4.zip /content/quantized/model-int4

print("✅ Zipping complete! Ready for download.")

  adding: content/quantized/model-int8/ (stored 0%)
  adding: content/quantized/model-int8/model.safetensors (deflated 14%)
  adding: content/quantized/model-int8/config.json (deflated 57%)
  adding: content/quantized/model-int8/generation_config.json (deflated 29%)
  adding: content/quantized/model-int4/ (stored 0%)
  adding: content/quantized/model-int4/model.safetensors (deflated 17%)
  adding: content/quantized/model-int4/config.json (deflated 57%)
  adding: content/quantized/model-int4/generation_config.json (deflated 29%)
✅ Zipping complete! Ready for download.
